# Fine tuning, for the n=4 case

Number 4; 

MRPC for parallel model 

In [1]:
import torch
import torch.nn as nn

# Needed for parallel 
from collections import OrderedDict

# For training 
from network_architecture_v2 import MyBertForSequenceClassification

# For fine tuning
from datasets import load_dataset #, load_metric
from transformers import BertTokenizer
from transformers import Trainer, TrainingArguments
import numpy as np

In [2]:
# Load dataset
dataset = load_dataset('glue', 'mrpc')

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"], 
        examples["sentence2"], 
        padding="max_length", 
        truncation=True,
        max_length=224
    )
    
tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


# Load the parallel model

This involves a bit more code

In [3]:
checkpoint_0 = torch.load('bert-save-4-continue-2/model_checkpoint_0_batch_idx=15000')
checkpoint_1 = torch.load('bert-save-4-continue-2/model_checkpoint_1_batch_idx=15000')
checkpoint_2 = torch.load('bert-save-4-continue-2/model_checkpoint_2_batch_idx=15000')
checkpoint_3 = torch.load('bert-save-4-continue-2/model_checkpoint_3_batch_idx=15000')

In [4]:
keys_0 = checkpoint_0['model_state_dict'].keys()
keys_1 = checkpoint_1['model_state_dict'].keys()
keys_2 = checkpoint_2['model_state_dict'].keys()
keys_3 = checkpoint_3['model_state_dict'].keys()

In [5]:
# Ugh, this is so dumb
new_dict = OrderedDict()
keys_0 = checkpoint_0['model_state_dict'].keys()
counter = 0
for key in keys_0:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        if int(split[2]) > counter:
            counter = int(split[2])
            
        split.insert(3, 'layer')
        new_key = '.'.join(split[1:])
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
    else:
        new_key = key
        if 'close_nsp' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_nsp'
            new_key = '.'.join(split)
        if 'close_mlm' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_mlm'
            new_key = '.'.join(split)
        
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
        
print(counter)

# Now for the remaining parts? 
keys_1 = checkpoint_1['model_state_dict'].keys()
new_counter = 0
for key in keys_1:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')

        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_1['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_1['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_2 = checkpoint_2['model_state_dict'].keys()
for key in keys_2:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_2['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_2['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_3 = checkpoint_3['model_state_dict'].keys()
for key in keys_3:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_3['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_3['model_state_dict'][key]

32
64
96


In [6]:
model_parallel = torch.load('serialnet_bert_128', weights_only=False)
# model_parallel.load_state_dict(new_dict)

In [7]:
training_parallel = MyBertForSequenceClassification(model_parallel)

# With weights loaded, go ahead and train

In [9]:
training_args = TrainingArguments(
    output_dir="./results_3",          # Output directory
    evaluation_strategy="epoch",    # Evaluate at the end of each epoch
    save_strategy="epoch",          # Save the model at the end of each epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=3,             # Number of epochs
    weight_decay=0.01,              # Weight decay
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=500,              # Log every 500 steps
    fp16=True,                      # Use mixed precision (if supported by hardware)
    load_best_model_at_end=True,    # Load the best model at the end of training
    dataloader_drop_last=True,  # Set to False for SST-2 (important for small batches)
)


/home/sjiang/braids_v3/pip-test/lib/python3.13/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}


trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.640531,0.685000
2,No log,0.624141,0.685000
3,0.647900,0.621670,0.685000


TrainOutput(global_step=687, training_loss=0.6438492021185862, metrics={'train_runtime': 388.8107, 'train_samples_per_second': 28.302, 'train_steps_per_second': 1.767, 'total_flos': 0.0, 'train_loss': 0.6438492021185862, 'epoch': 3.0})

In [12]:
.647900 - .636800

0.011099999999999999

# QNLI

In [8]:
# Load QNLI dataset
dataset = load_dataset('glue', 'qnli')

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["question"], 
        examples["sentence"], 
        padding="max_length", 
        truncation=True,
        max_length=224
    )

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Rename the label column to "labels" for compatibility with Hugging Face Trainer
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set format for PyTorch tensors
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [9]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results_4",          # Output directory
    evaluation_strategy="epoch",         # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save the model at the end of each epoch
    learning_rate=2e-5,                  # Learning rate
    per_device_train_batch_size=16,      # Batch size for training
    per_device_eval_batch_size=16,       # Batch size for evaluation
    num_train_epochs=2,                  # Number of epochs
    weight_decay=0.01,                   # Weight decay
    logging_dir="./logs4",                # Directory for storing logs
    logging_steps=500,                   # Log every 500 steps
    fp16=True,                           # Use mixed precision (if supported by hardware)
    load_best_model_at_end=True,         # Load the best model at the end of training
    dataloader_drop_last=True,           # Drop last batch if it's incomplete
)


/home/sjiang/braids_v3/pip-test/lib/python3.13/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}
    
# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.669800,0.660446,0.602823


TrainOutput(global_step=6546, training_loss=0.6754528323044443, metrics={'train_runtime': 3274.5877, 'train_samples_per_second': 31.987, 'train_steps_per_second': 1.999, 'total_flos': 0.0, 'train_loss': 0.6754528323044443, 'epoch': 1.0})

In [14]:

trainer.evaluate()

{'eval_loss': 0.660446286201477,
 'eval_accuracy': 0.6028226017951965,
 'eval_runtime': 47.1205,
 'eval_samples_per_second': 115.937,
 'eval_steps_per_second': 7.258,
 'epoch': 1.0}

In [15]:
0.602823 - 0.590542

0.012280999999999986